# Behavior-Aware SASRec

Train and compare:
- **purchase_only**: SASRec on purchase sequences only
- **behavior_aware**: same SASRec backbone + cart add/remove behavior embeddings

Target is always the **next purchase**. Cart events are used as context before the target purchase.

This notebook calls the package in `behavior_aware_sasrec/ds5500_recsys/`.


## 0. Setup

In [ ]:
from pathlib import Path
import sys
import json

import torch

# Make the local package importable whether you launch from repo root or this folder
NOTEBOOK_DIR = Path.cwd()
CANDIDATES = [NOTEBOOK_DIR, NOTEBOOK_DIR / "behavior_aware_sasrec", NOTEBOOK_DIR.parent]
for p in CANDIDATES:
    if (p / "ds5500_recsys").exists():
        PKG_ROOT = p
        break
else:
    raise FileNotFoundError(
        "Could not find ds5500_recsys/. Open this notebook from the repo root "
        "or from behavior_aware_sasrec/."
    )

sys.path.insert(0, str(PKG_ROOT))
print("Package root:", PKG_ROOT.resolve())
print("Torch:", torch.__version__, "| device available:", "cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
from ds5500_recsys.data.sequences import (
    build_user_sequences,
    load_behavior_events,
    make_synthetic_events,
    temporal_split_sequences,
)
from ds5500_recsys.metrics import format_metrics
from ds5500_recsys.train import TrainConfig, train_model


## 1. Config

Toggle `USE_SYNTHETIC` for a quick dry-run without parquet files.  
For the real experiment, set `USE_SYNTHETIC = False` and point `DATA_DIR` at your Step 1 subsample.


In [ ]:
# --- data ---
USE_SYNTHETIC = True          # True: fake data smoke test; False: Synerise subsample
DATA_DIR = Path("data/subsampled")  # relative to repo root when USE_SYNTHETIC=False
TOP_N_ITEMS = 5000            # None = keep all items; global purchase-frequency pool
MIN_BUYS = 2

# --- model / train ---
MODE = "both"                 # "purchase_only" | "behavior_aware" | "both"
MAX_LEN = 50
HIDDEN_SIZE = 64
N_BLOCKS = 2
N_HEADS = 1
DROPOUT = 0.2
BATCH_SIZE = 128
LR = 1e-3
EPOCHS = 3 if USE_SYNTHETIC else 20
N_NEGATIVES = 1
EVAL_NEGATIVES = 100
PATIENCE = 5
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

OUTPUT_DIR = PKG_ROOT / "artifacts" / ("notebook_synthetic" if USE_SYNTHETIC else "notebook_run")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("USE_SYNTHETIC =", USE_SYNTHETIC)
print("DEVICE =", DEVICE)
print("OUTPUT_DIR =", OUTPUT_DIR)


## 2. Load events

In [ ]:
INCLUDE = ("product_buy", "add_to_cart", "remove_from_cart")

if USE_SYNTHETIC:
    events = make_synthetic_events(n_users=300, n_items=150, seed=SEED)
    print("Synthetic events generated")
else:
    # try a few common locations
    data_candidates = [
        DATA_DIR,
        Path("data/subsampled"),
        Path("../data/subsampled"),
        Path("Desktop/DS5500/data/subsampled"),
        Path.home() / "Desktop/DS5500/data/subsampled",
    ]
    data_dir = next((p for p in data_candidates if p.exists()), None)
    if data_dir is None:
        raise FileNotFoundError(
            "No subsampled data found. Run DataIngestion_Subsample.ipynb first, "
            "or set USE_SYNTHETIC=True."
        )
    print("Loading from:", data_dir.resolve())
    events = load_behavior_events(data_dir, include_behaviors=INCLUDE)

print(f"Rows: {len(events):,}")
display(events["behavior"].value_counts())
events.head()


## 3. Build user sequences + temporal split

Per user with ≥2 purchases: last purchase → test target; second-to-last → val (if available); earlier purchases → train instances. History includes events **before** the target purchase (cart signal allowed, no target leakage).


In [ ]:
payload = build_user_sequences(
    events,
    include_behaviors=INCLUDE,
    min_buys=MIN_BUYS,
    max_len=None,
    top_n_items=None if USE_SYNTHETIC else TOP_N_ITEMS,
)
print(f"Users={payload['n_users']:,} | items={payload['n_items']:,} | sequences={len(payload['sequences']):,}")

splits = temporal_split_sequences(payload["sequences"])
print(
    f"Split users — train: {len(splits['train']):,} | "
    f"val: {len(splits['val']):,} | test: {len(splits['test']):,}"
)


## 4. Train config

In [ ]:
cfg = TrainConfig(
    hidden_size=HIDDEN_SIZE,
    max_len=MAX_LEN,
    n_blocks=N_BLOCKS,
    n_heads=N_HEADS,
    dropout=DROPOUT,
    batch_size=BATCH_SIZE,
    lr=LR,
    epochs=EPOCHS,
    n_negatives=N_NEGATIVES,
    eval_negatives=EVAL_NEGATIVES,
    seed=SEED,
    device=DEVICE,
    patience=PATIENCE,
)
cfg


## 5. Train models

`behavior_aware` uses buy + add-to-cart + remove-from-cart in history.  
`purchase_only` strips cart events but uses the **same purchase targets**.


In [ ]:
modes = ["purchase_only", "behavior_aware"] if MODE == "both" else [MODE]
results = {}

for mode in modes:
    print(f"\n=== Training {mode} ===")
    result = train_model(mode, payload, splits, cfg, OUTPUT_DIR / mode)
    results[mode] = {
        "test_metrics": result["test_metrics"],
        "best_val": result["best_val"],
        "best_checkpoint": result["best_checkpoint"],
    }
    print(f"[{mode}] final test:", format_metrics(result["test_metrics"]))

results


## 6. Compare results

In [ ]:
import pandas as pd

rows = []
for mode, payload_res in results.items():
    m = payload_res["test_metrics"]
    rows.append({
        "model": mode,
        "HR@10": m.get("HR@10"),
        "NDCG@10": m.get("NDCG@10"),
        "MRR": m.get("MRR"),
        "HR@5": m.get("HR@5"),
        "NDCG@5": m.get("NDCG@5"),
        "n": int(m.get("n", 0)),
    })

comparison = pd.DataFrame(rows).set_index("model")
comparison


In [ ]:
summary_path = OUTPUT_DIR / "summary.json"
with summary_path.open("w") as f:
    json.dump(results, f, indent=2)
print("Wrote", summary_path.resolve())


## Notes

- Synthetic metrics are only for pipeline checking; report real-cohort numbers for the project.
- Keep the same `TOP_N_ITEMS` for both models so the comparison stays fair.
- Checkpoints: `artifacts/.../<mode>/<mode>_best.pt`
